# 02 — Préparation du jeu de données MediQAI

## Objectif du notebook

Ce notebook a pour objectif de préparer les données médicales francophones de **MediQAI** afin de construire un benchmark homogène et directement exploitable pour l'évaluation des grands modèles de langage.

À partir des fichiers Parquet obtenus lors de l'exploration du dataset, les principales étapes de préparation sont :

1. charger les splits `train`, `validation` et `test` ;
2. vérifier leur structure ;
3. effectuer un nettoyage léger des variables textuelles ;
4. structurer les propositions des questions à choix multiple ;
5. construire la réponse de référence ;
6. générer un contexte textuel standardisé pour chaque question ;
7. effectuer plusieurs contrôles de qualité et de cohérence ;
8. enregistrer les jeux de données préparés au format Parquet.

La préparation porte sur la configuration **MCQU** (*Multiple Choice Question Unique*), sélectionnée à l'issue de l'exploration précédente. Chaque question possédant une réponse correcte unique, cette configuration permet de comparer directement la prédiction d'un LLM à une vérité terrain clairement définie.

Les splits officiels `train`, `validation` et `test` sont conservés afin de permettre leur utilisation séparée dans les différentes étapes du projet.

## 1. Initialisation de l'environnement

Les bibliothèques nécessaires à la manipulation des données sont importées et les chemins vers les données brutes et préparées sont définis.

Les fichiers issus de l'exploration précédente sont lus depuis `data/raw/medical`, tandis que les données préparées seront enregistrées dans `data/processed`.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
# définir le chemin d'importation et d'enregistrement des données
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "medical"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Chargement des données MCQU

Les trois splits officiels de la configuration MCQU (`train`, `validation` et `test`) sont chargés séparément.

Cette séparation est conservée pendant toute la préparation afin de pouvoir identifier explicitement l'origine de chaque observation.

In [3]:
splits = ["train", "validation", "test"]

dfs_rw = {
    split: pd.read_parquet(
        RAW_DIR / f"medical_mcqu_{split}.parquet"
    )
    for split in splits
}

## 3. Vérification de la structure des données

Avant d'appliquer les transformations, les dimensions et les types de variables de chaque split sont contrôlés.

Cette étape permet de vérifier que les trois jeux de données possèdent une structure compatible avec le pipeline de préparation et d'identifier d'éventuelles différences de schéma avant traitement.

In [4]:
# vérification des dimensions des DataFrames
for split, df in dfs_rw.items():
    print(f"{split.capitalize()} set shape: {df.shape}")

Train set shape: (10113, 14)
Validation set shape: (2561, 14)
Test set shape: (4343, 14)


In [5]:
# vérification du schéma des DataFrames
for split, df in dfs_rw.items():
    print(f"\n{split.capitalize()} set schema:")
    print(df.dtypes)


Train set schema:
id                       object
clinical_case            object
question                 object
answer_a                 object
answer_b                 object
answer_c                 object
answer_d                 object
answer_e                 object
correct_answers          object
task                     object
medical_subject          object
question_type            object
question_length_chars     int64
question_length_words     int64
dtype: object

Validation set schema:
id                       object
clinical_case            object
question                 object
answer_a                 object
answer_b                 object
answer_c                 object
answer_d                 object
answer_e                 object
correct_answers          object
task                     object
medical_subject          object
question_type            object
question_length_chars     int64
question_length_words     int64
dtype: object

Test set schema:
id             

## 4. Nettoyage des variables textuelles

Un nettoyage léger est appliqué aux variables textuelles afin d'homogénéiser leur format sans modifier leur contenu médical.

La fonction `clean_text_column` :

- remplace les valeurs manquantes par une chaîne vide ;
- convertit les valeurs en chaînes de caractères ;
- normalise les espaces successifs ;
- supprime les espaces inutiles en début et en fin de texte.

Les valeurs manquantes de `clinical_case` ne sont pas considérées comme des observations invalides : certaines questions peuvent être formulées sans cas clinique associé. Elles sont donc conservées et représentées par une chaîne vide.

In [6]:


def clean_text_column(series):
    return (
        series
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

## 5. Structuration des propositions de réponse

Les questions MCQU comportent jusqu'à cinq propositions de réponse stockées dans les colonnes `answer_a` à `answer_e`.

Un dictionnaire `CHOICE_COLUMNS` établit la correspondance entre les lettres **A–E** et les colonnes correspondantes.

Deux représentations complémentaires sont ensuite construites :

- `choices` : représentation structurée des propositions sous forme de dictionnaire ;
- `choices_text` : représentation textuelle formatée destinée à être intégrée au contexte envoyé aux LLM.

Exemple de représentation textuelle :

A. Première proposition  
B. Deuxième proposition  
C. Troisième proposition  
D. Quatrième proposition  
E. Cinquième proposition

In [7]:
CHOICE_COLUMNS = {
    "A": "answer_a",
    "B": "answer_b",
    "C": "answer_c",
    "D": "answer_d",
    "E": "answer_e",
}

def build_choices(row):
    return {
        letter: row[column]
        for letter, column in CHOICE_COLUMNS.items()
    }

# dont une destinée au prompt 
def format_choices(row):
    return "\n".join(
        f"{letter}. {row[column]}"
        for letter, column in CHOICE_COLUMNS.items()
    )

## 6. Construction de la réponse de référence

MediQAI fournit la lettre de la réponse correcte dans la variable `correct_answers`.

Pour faciliter l'évaluation ultérieure des modèles, deux informations de référence sont conservées :

- `reference_letter` : lettre correspondant à la réponse correcte ;
- `reference_answer` : contenu textuel de la proposition correspondante.

La fonction `get_reference_answer` utilise la lettre de référence pour retrouver automatiquement le texte de la réponse dans les colonnes `answer_a` à `answer_e`.

Cette double représentation permettra notamment de comparer directement la lettre prédite par un LLM à la lettre de référence.

In [8]:
def get_reference_answer(row):
    letter = row["correct_answers"].strip().upper()
    column = CHOICE_COLUMNS.get(letter)

    if column is None:
        return None

    return row[column]

## 7. Construction du contexte de la question

Afin de fournir aux différents LLM une entrée homogène, les informations nécessaires à la résolution de chaque question sont regroupées dans une variable unique appelée `question_context`.

Le contexte est construit à partir de :

1. du cas clinique, lorsqu'il est disponible ;
2. de la question ;
3. des propositions de réponse.

Le cas clinique est volontairement omis lorsqu'il est absent afin de ne pas introduire de contenu artificiel.

Le format obtenu est par exemple :

> **Cas clinique :**  
> Un homme de 63 ans est hospitalisé...
>
> **Question :**  
> Pour confirmer le diagnostic de maladie d'Addison...
>
> **Propositions :**  
> A. Test de stimulation...  
> B. Test de freinage...  
> C. ...

Cette représentation constituera le contenu médical injecté dans les prompts lors de l'interrogation des LLM.

In [9]:

def build_question_context(row):
    parts = []

    if row["clinical_case"]:
        parts.append(f"Cas clinique :\n{row['clinical_case']}")

    parts.append(f"Question :\n{row['question']}")
    parts.append(f"Propositions :\n{row['choices_text']}")

    return "\n\n".join(parts)

## 8. Construction du dataset préparé

La fonction `prepare_mcqu` regroupe les différentes étapes de préparation afin d'appliquer exactement le même traitement aux trois splits.

Pour chaque observation, elle :

- nettoie les variables textuelles ;
- normalise la lettre de référence ;
- construit les représentations `choices` et `choices_text` ;
- récupère la réponse textuelle de référence ;
- construit `question_context` ;
- ajoute le split et la configuration ;
- génère un identifiant unique `sample_id`.

L'identifiant suit la convention :

`mcqu_<split>_<id>`

Par exemple : `mcqu_validation_25168`.

Cette convention permettra de retrouver précisément chaque question dans les résultats expérimentaux produits ultérieurement.

In [10]:
def prepare_mcqu(df, split):
    prepared = df.copy()

    text_columns = [
        "clinical_case",
        "question",
        "answer_a",
        "answer_b",
        "answer_c",
        "answer_d",
        "answer_e",
        "medical_subject",
        "question_type",
        "task",
    ]

    for column in text_columns:
        prepared[column] = clean_text_column(
            prepared[column]
        )

    prepared["reference_letter"] = (
        prepared["correct_answers"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    prepared["choices"] = prepared.apply(
        build_choices,
        axis=1
    )

    prepared["choices_text"] = prepared.apply(
        format_choices,
        axis=1
    )

    prepared["reference_answer"] = prepared.apply(
        get_reference_answer,
        axis=1
    )

    prepared["question_context"] = prepared.apply(
        build_question_context,
        axis=1
    )

    prepared["split"] = split
    prepared["configuration"] = "mcqu"

    prepared["sample_id"] = (
        "mcqu_"
        + split
        + "_"
        + prepared["id"].astype(str)
    )

    return prepared

In [11]:
# application au jeu de données complet
dfs_processed = {
    split: prepare_mcqu(df, split)
    for split, df in dfs_rw.items()
}

## 9. Sélection des variables finales

Après préparation, seules les variables utiles au benchmark et à l'analyse ultérieure sont conservées.

Le dataset final contient notamment :

- les identifiants de l'observation (`sample_id`, `id`) ;
- la configuration et le split ;
- les données originales de la question ;
- les propositions de réponse ;
- la lettre et le texte de la réponse de référence ;
- les métadonnées médicales ;
- les représentations préparées (`choices`, `choices_text`, `question_context`).

Les variables dérivées permettent de conserver à la fois une représentation structurée des données pour l'analyse et une représentation textuelle directement exploitable pour les futurs prompts.


In [12]:
final_columns = [
    "sample_id",
    "id",
    "configuration",
    "split",
    "clinical_case",
    "question",
    "answer_a",
    "answer_b",
    "answer_c",
    "answer_d",
    "answer_e",
    "choices",
    "choices_text",
    "reference_letter",
    "reference_answer",
    "medical_subject",
    "question_type",
    "task",
    "question_context",
]

# avec :
# - "configuration" : le type de question (ici "mcqu" pour "multiple choice question")
# - "choices" : un dictionnaire des propositions de réponses
# - "choices_text" : une chaîne de caractère formattée des propositions de réponses.
# - "question_context" : une chaîne de caractère formattée du contexte de la question, incluant le cas clinique, la question et les propositions" 


In [13]:
for split in dfs_processed:
    dfs_processed[split] = (
        dfs_processed[split][final_columns]
    )

## 10. Contrôles de qualité et de cohérence

Plusieurs contrôles sont réalisés avant l'enregistrement du benchmark afin de détecter d'éventuelles anomalies susceptibles d'affecter les expérimentations.

Les vérifications portent sur :

- l'unicité des `sample_id` au sein de chaque split ;
- la validité des lettres de référence, qui doivent appartenir à `{A, B, C, D, E}` ;
- l'absence de réponses de référence manquantes ;
- le chevauchement éventuel des identifiants entre les splits ;
- la présence de contextes de questions strictement identiques au sein d'un même split.

Ces contrôles permettent de vérifier l'intégrité du benchmark avant son utilisation pour l'évaluation des LLM.

In [14]:
# unicité des identifiants 
for split, df in dfs_processed.items():
    if df["sample_id"].duplicated().any():
        raise ValueError(f"Duplicate sample_id found in {split} set.")

In [15]:
# validité des réponses
valid_letters = {"A", "B", "C", "D", "E"}

for split, df in dfs_processed.items():
    invalid = ~df["reference_letter"].isin(valid_letters)

    print(
        split,
        "réponses invalides :",
        invalid.sum()
    )

train réponses invalides : 0
validation réponses invalides : 0
test réponses invalides : 0


In [16]:
# Réponse de référence manquante
for split, df in dfs_processed.items():
    print(
        split,
        "références manquantes :",
        df["reference_answer"].isna().sum()
    )

train références manquantes : 0
validation références manquantes : 0
test références manquantes : 0


In [17]:
# vérification de l'unicité des identifiants entre les splits
train_ids = set(dfs_processed["train"]["id"])
validation_ids = set(dfs_processed["validation"]["id"])
test_ids = set(dfs_processed["test"]["id"])

print(
    "ID validation présents dans train :",
    len(train_ids & validation_ids)
)

print(
    "ID test présents dans train :",
    len(train_ids & test_ids)
)

print(
    "ID test présents dans validation :",
    len(validation_ids & test_ids)
)

ID validation présents dans train : 0
ID test présents dans train : 0
ID test présents dans validation : 0


### Vérification des chevauchements entre les splits

La comparaison des identifiants permet de détecter si certaines observations apparaissent dans plusieurs splits.

La présence d'identifiants communs ne doit pas être ignorée : elle peut indiquer un chevauchement entre les partitions du dataset et doit être prise en compte lors de la définition du protocole expérimental, en particulier si les différents splits sont utilisés pour le développement, la sélection de paramètres et l'évaluation finale.

Ce contrôle est donc conservé explicitement afin d'éviter d'interpréter les splits comme totalement indépendants sans vérification préalable.

In [19]:
# Recherche des doublons exacts de question_context
# à l'intérieur de chaque split

for split, df in dfs_processed.items():
    duplicate_mask = df["question_context"].duplicated(
        keep=False
    )

    duplicate_rows = df.loc[duplicate_mask]

    print(
        f"{split} : "
        f"{duplicate_mask.sum()} lignes concernées par un doublon exact, "
        f"correspondant à "
        f"{duplicate_rows['question_context'].nunique()} contextes distincts"
    )

train : 167 lignes concernées par un doublon exact, correspondant à 81 contextes distincts
validation : 8 lignes concernées par un doublon exact, correspondant à 4 contextes distincts
test : 63 lignes concernées par un doublon exact, correspondant à 31 contextes distincts


## 11. Enregistrement du benchmark préparé

Après validation, chacun des trois splits est enregistré séparément au format **Parquet** dans le répertoire `data/processed`.

Les fichiers produits suivent la convention :

- `benchmark_mcqu_train.parquet`
- `benchmark_mcqu_validation.parquet`
- `benchmark_mcqu_test.parquet`

Ces fichiers constituent désormais les données de référence préparées qui seront utilisées dans les étapes expérimentales du projet.

In [18]:
for split, df in dfs_processed.items():
    output_path = (
        PROCESSED_DIR
        / f"benchmark_mcqu_{split}.parquet"
    )

    df.to_parquet(
        output_path,
        index=False
    )

    print(
        f"{split}: {len(df)} lignes "
        f"enregistrées dans {output_path}"
    )

train: 10113 lignes enregistrées dans c:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale\data\processed\benchmark_mcqu_train.parquet
validation: 2561 lignes enregistrées dans c:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale\data\processed\benchmark_mcqu_validation.parquet
test: 4343 lignes enregistrées dans c:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale\data\processed\benchmark_mcqu_test.parquet


## Conclusion

Ce notebook a permis de transformer les données MCQU brutes de MediQAI en un benchmark standardisé destiné à l'évaluation des modèles de langage.

La préparation conserve le contenu médical original tout en ajoutant les représentations nécessaires au protocole expérimental : propositions structurées, lettre de référence, réponse textuelle de référence, contexte complet de la question et identifiant unique.

Des contrôles de qualité ont également été réalisés afin d'identifier les réponses invalides, les références manquantes, les doublons et les éventuels chevauchements entre les splits.

Les fichiers Parquet produits constituent ainsi une base homogène pour l'étape suivante du projet : **la construction et l'évaluation des prompts utilisés pour interroger les différents LLM**.